In [2]:
import warnings
from reportlab.lib.pagesizes import portrait, A4
from reportlab.lib.units import mm
from reportlab.pdfgen import canvas
from reportlab.lib.styles import ParagraphStyle
from reportlab.platypus import Paragraph
import re
import pandas as pd
warnings.filterwarnings('ignore')

In [3]:
master_file_path = "/Users/aswathshakthi/PycharmProjects/MLMR/MNP26/Result/1_Master_Records_Seat_Seq.xlsx"

In [4]:
data = pd.read_excel(master_file_path)
data

,Application Number,Names,Beneficiary Name,Requested Item,Handicapped Status,Gender,Gender Category,Beneficiary Type,Item Type,Article Category,Super Category Article,Requested Item Tk,Quantity,Total Value,Waiting Hall Quantity,Token Quantity,Sequence No,R_Names
0,D001,Andaman,Andaman,Bosch Electrician Kit 10 Re,NaN,NaN,NaN,District,Article,Electricals,Livelihood Aid,Bosch Elec Kit 10 Re,1,4307.0,0,1,49,Andaman
1,D001,Andaman,Andaman,Gp Welding Machine Arc 200,NaN,NaN,NaN,District,Article,Electricals,Livelihood Aid,Gp Welding Arc 200,1,6726.0,0,1,53,Andaman
2,D002,Ariyalur,Ariyalur,Agri Battery Sprayer,NaN,NaN,NaN,District,Article,Agriculture,Agricultural Welfare Aid,Agri Battery Sprayer,8,33600.0,8,0,43,Ariyalur
3,D002,Ariyalur,Ariyalur,Education Aid,NaN,NaN,NaN,District,Aid,Aid,Education Aid,Education Aid,7,70000.0,6,1,33,Ariyalur
4,D002,Ariyalur,Ariyalur,Iron Box Brass,NaN,NaN,NaN,District,Article,Electronics,Livelihood Aid,Iron Box Brass,1,7500.0,1,0,55,Ariyalur
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
994,P455,P455 - Karuppi,Karuppi,Ex Gratia Aid,No,Female,Widow,Public,Aid,Aid,Medical Aid,Ex Gratia Aid,1,100000.0,0,1,31,AA_Public
995,P456,P456 - S.Ranjith Singh,S.Ranjith Singh,Gents Cycle,No,Male,NaN,Public,Article,Bicycle & Tricycle,Student Welfare Aid,Gents Cycle,1,5175.0,0,1,94,AA_Public
996,P457,P457 - S.Lathika,S.Lathika,Laptop,No,Female,Unmarried,Public,Article,Computers & Printers,Student Welfare Aid,Laptop,1,40000.0,0,1,22,AA_Public
997,P458,P458 - V.Raman,V.Raman,Ex Gratia Aid,No,Male,NaN,Public,Aid,Aid,Medical Aid,Ex Gratia Aid,1,100000.0,0,1,31,AA_Public


In [5]:
# Custom Beneficiary Type order
benef_order = ["Institutions", "Public", "District"]
data["Beneficiary Type"] = pd.Categorical(
    data["Beneficiary Type"],
    categories=benef_order,
    ordered=True
)

# Handicapped first
handicap_yes = {"Yes"}
data["_handicap_first"] = (
    data["Handicapped Status"].astype(str).str.strip().str.lower().isin(handicap_yes)
    .map({True: 0, False: 1})
)

# Put P116 first only inside Public + Laptop rows
is_public_laptop = (
    data["Beneficiary Type"].astype(str).eq("Public")
    & data["Requested Item"].astype(str).str.contains("Laptop", case=False, na=False)
)
data["_p116_top"] = 1
data.loc[is_public_laptop & data["Application Number"].astype(str).eq("P116"), "_p116_top"] = 0

# Final sort
data = data.sort_values(
    by=["Sequence No", "Beneficiary Type", "_p116_top", "_handicap_first", "Names"],
    ascending=[True, True, True, True, True],
    ignore_index=True
).drop(columns=["_p116_top", "_handicap_first"])

data


,Application Number,Names,Beneficiary Name,Requested Item,Handicapped Status,Gender,Gender Category,Beneficiary Type,Item Type,Article Category,Super Category Article,Requested Item Tk,Quantity,Total Value,Waiting Hall Quantity,Token Quantity,Sequence No,R_Names
0,P435,P435 - B.Senthil Kumar,B.Senthil Kumar,Cow & Calf,No,Male,NaN,Public,Article,Livestock,Livelihood Aid,Cattle,1,45000.0,0,1,1,AA_Public
1,P049,P049 - Valli.M,Valli.M,Goat(1 Pair),No,Female,Married,Public,Article,Livestock,Livelihood Aid,Goat,1,10000.0,0,1,2,AA_Public
2,P123,P123 - Viruthambal.M,Viruthambal.M,Goat(1 Pair),Yes,Female,Married,Public,Article,Livestock,Livelihood Aid,Goat,1,10000.0,0,1,2,AA_Public
3,P124,P124 - Madurai.D,Madurai.D,Goat(1 Pair),Yes,Male,NaN,Public,Article,Livestock,Livelihood Aid,Goat,1,10000.0,0,1,2,AA_Public
4,P125,P125 - Mariyammal.M,Mariyammal.M,Goat(1 Pair),No,Female,Widow,Public,Article,Livestock,Livelihood Aid,Goat,1,10000.0,0,1,2,AA_Public
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
994,I013,I013 - Polambakkam Govt Higher Secondary School.,Polambakkam Govt Higher Secondary School.,Steel Cupboard 6 1/2',NaN,NaN,NaN,Institutions,Article,Furnitures,Infrastructure / Institutional Support Aid,Steel Cupboard,1,12390.0,1,0,120,A_Institutions
995,I002,"I002 - Athivakkam,Panchayat Union Primary School.","Athivakkam,Panchayat Union Primary School.",Steel cupboard Glass Book Shelf,NaN,NaN,NaN,Institutions,Article,Furnitures,Student Welfare Aid,Glass Book Shelf,1,14750.0,1,0,121,A_Institutions
996,I012,I012 - Avanippur Government Higher Secondary S...,Avanippur Government Higher Secondary School,Ceiling Fan,NaN,NaN,NaN,Institutions,Article,Electricals,Infrastructure / Institutional Support Aid,Ceiling Fan,10,15000.0,10,0,122,A_Institutions
997,I013,I013 - Polambakkam Govt Higher Secondary School.,Polambakkam Govt Higher Secondary School.,Ceiling Fan,NaN,NaN,NaN,Institutions,Article,Electricals,Infrastructure / Institutional Support Aid,Ceiling Fan,5,7500.0,5,0,122,A_Institutions


In [6]:
## Baneficiary name change
rename_dict = {
"I001 - Government Leprosy Centre,Chengalpattu.":"I001-Govt Leprosy Centre,CGL",
"I002 - Athivakkam,Panchayat Union Primary School.":"I002-Athivakkam,Panchayat School",
"I003 - Thirukazhukundram,Govt Girls Higher Secondary School.":"I003-Thirukazhukundram,Govt Girls School",
"I004 - Acharapakkam,Govt Girls Higher Secondary School.":"I004-Acharapakkam,Govt Girls School",
"I005 - Maduranthagam,District Educational Office.":"I005-Maduranthagam,District Edu Off",
"I006 - Thozhupedu,Govt Higher Secondary School.":"I006-Thozhupedu,Govt School",
"I007 - Kayappakkam,Government Higher Secondary School.":"I007-Kayappakkam,Government School",
"I008 - Nolambur Government Higher SecondarySchool":"I008-Nolambur Government School",
"I009 - Acharapakkam, Govt Boys Higher Secondary School.":"I009-Acharapakkam,Govt Boys School",
"I010 - Cheyyur Govt Girls Higher Secondary School.":"I010-Cheyyur Govt Girls School",
"I011 - Chunambedu Govt Higher Secondary School.":"I011-Chunambedu Govt School",
"I012 - Avanippur Government Higher Secondary School":"I012-Avanippur Government School",
"I013 - Polambakkam Govt Higher Secondary School.":"I013-Polambakkam Govt School",
}

# Rename values in df["Names"]
data["Names"] = data["Names"].replace(rename_dict)

##### 5.2 - Repeating Sequence Number

In [7]:
list = data["Sequence No"].unique()
Note=pd.DataFrame()
for i in list:

    if data[data["Sequence No"]==i]["Requested Item"].nunique() == 1:
        pass
    else:
        Note = Note.append([("Sequence "+str(i))],ignore_index=True)
if Note.empty:
    print("None")
else:
    print(Note)

None


##### 5.2 - Repeating Articles

In [8]:
list = data["Requested Item"].unique()
Note=pd.DataFrame()
for i in list:

    if data[data["Requested Item"]==i]["Sequence No"].nunique() == 1:
        pass
    else:
        Note = Note.append([i],ignore_index=True)
if Note.empty:
    print("None")
else:
    print(Note[0])

None


##### 5.3 - Check for Missing Sequence Number

In [9]:
uqs = data["Sequence No"].unique()
for i in range(1,uqs.max()):
    if i not in uqs:
        print("Missing Sequence : ",i)

Missing Sequence :  21


##### 5.4 Remove Duplicated Rows

In [10]:
data.dropna(how='all')
# data["Token Quantity"] = data["Quantity"]

,Application Number,Names,Beneficiary Name,Requested Item,Handicapped Status,Gender,Gender Category,Beneficiary Type,Item Type,Article Category,Super Category Article,Requested Item Tk,Quantity,Total Value,Waiting Hall Quantity,Token Quantity,Sequence No,R_Names
0,P435,P435 - B.Senthil Kumar,B.Senthil Kumar,Cow & Calf,No,Male,NaN,Public,Article,Livestock,Livelihood Aid,Cattle,1,45000.0,0,1,1,AA_Public
1,P049,P049 - Valli.M,Valli.M,Goat(1 Pair),No,Female,Married,Public,Article,Livestock,Livelihood Aid,Goat,1,10000.0,0,1,2,AA_Public
2,P123,P123 - Viruthambal.M,Viruthambal.M,Goat(1 Pair),Yes,Female,Married,Public,Article,Livestock,Livelihood Aid,Goat,1,10000.0,0,1,2,AA_Public
3,P124,P124 - Madurai.D,Madurai.D,Goat(1 Pair),Yes,Male,NaN,Public,Article,Livestock,Livelihood Aid,Goat,1,10000.0,0,1,2,AA_Public
4,P125,P125 - Mariyammal.M,Mariyammal.M,Goat(1 Pair),No,Female,Widow,Public,Article,Livestock,Livelihood Aid,Goat,1,10000.0,0,1,2,AA_Public
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
994,I013,I013-Polambakkam Govt School,Polambakkam Govt Higher Secondary School.,Steel Cupboard 6 1/2',NaN,NaN,NaN,Institutions,Article,Furnitures,Infrastructure / Institutional Support Aid,Steel Cupboard,1,12390.0,1,0,120,A_Institutions
995,I002,"I002-Athivakkam,Panchayat School","Athivakkam,Panchayat Union Primary School.",Steel cupboard Glass Book Shelf,NaN,NaN,NaN,Institutions,Article,Furnitures,Student Welfare Aid,Glass Book Shelf,1,14750.0,1,0,121,A_Institutions
996,I012,I012-Avanippur Government School,Avanippur Government Higher Secondary School,Ceiling Fan,NaN,NaN,NaN,Institutions,Article,Electricals,Infrastructure / Institutional Support Aid,Ceiling Fan,10,15000.0,10,0,122,A_Institutions
997,I013,I013-Polambakkam Govt School,Polambakkam Govt Higher Secondary School.,Ceiling Fan,NaN,NaN,NaN,Institutions,Article,Electricals,Infrastructure / Institutional Support Aid,Ceiling Fan,5,7500.0,5,0,122,A_Institutions


##### Remove data where the `Change QTY` is 0 and add columns `Start Token No.` and `End Token No.` with default values as 0, since we wont generate token for them and these are used for merging with the generated file next stage.

In [11]:
# Zero_data = data[data["Token Quantity"]==0].reset_index(drop=True)
# Zero_data["Start Token No."] = int(0)  # create new column called Start Token No.
# Zero_data["End Token No."] = int(0)   # create new column called End Token No.
# Zero_data["Token Print for ARTL"] = int(0)
# Zero_data

In [12]:
# Zero_data["Quantity"].sum()

#### 6. **Generate token**: Generates token numbers for each item based on the quantity.

Filter the data to include only rows where the `Change QTY` is not 0. Then, add two new columns named `Start Token No.` and `End Token No.` with initial values set to 0.

For token generation:
1. In the first row, set the `Start Token No.` to 1.
2. Calculate the `End Token No.` by multiplying the `Change QTY` with 1 (as it's the first row).
3. For subsequent rows:
   - Start the `Start Token No.` with an increment of the previous row's `End Token No.`.
   - Calculate the `End Token No.` as the sum of the current row's `Change QTY` and the current row's `Start Token No.`, minus 1.

This process ensures that tokens are generated sequentially, with each row contributing to the token range based on the `Change QTY`.


In [13]:
# after your custom sort is done
data = data.reset_index(drop=True).copy()

tq = data["Token Quantity"].fillna(0).astype(int).clip(lower=0)

# running token end for non-zero rows
running_end = tq.cumsum()

# start/end: zero when Token Quantity == 0
data["Start Token No."] = (running_end - tq + 1).where(tq > 0, 0).astype(int)
data["End Token No."]   = running_end.where(tq > 0, 0).astype(int)

# optional: print flag for zero-token rows
data["Token Print for ARTL"] = data.get("Token Print for ARTL", 0)
data.loc[tq == 0, "Token Print for ARTL"] = 0
data

,Application Number,Names,Beneficiary Name,Requested Item,Handicapped Status,Gender,Gender Category,Beneficiary Type,Item Type,Article Category,...,Requested Item Tk,Quantity,Total Value,Waiting Hall Quantity,Token Quantity,Sequence No,R_Names,Start Token No.,End Token No.,Token Print for ARTL
0,P435,P435 - B.Senthil Kumar,B.Senthil Kumar,Cow & Calf,No,Male,NaN,Public,Article,Livestock,...,Cattle,1,45000.0,0,1,1,AA_Public,1,1,0
1,P049,P049 - Valli.M,Valli.M,Goat(1 Pair),No,Female,Married,Public,Article,Livestock,...,Goat,1,10000.0,0,1,2,AA_Public,2,2,0
2,P123,P123 - Viruthambal.M,Viruthambal.M,Goat(1 Pair),Yes,Female,Married,Public,Article,Livestock,...,Goat,1,10000.0,0,1,2,AA_Public,3,3,0
3,P124,P124 - Madurai.D,Madurai.D,Goat(1 Pair),Yes,Male,NaN,Public,Article,Livestock,...,Goat,1,10000.0,0,1,2,AA_Public,4,4,0
4,P125,P125 - Mariyammal.M,Mariyammal.M,Goat(1 Pair),No,Female,Widow,Public,Article,Livestock,...,Goat,1,10000.0,0,1,2,AA_Public,5,5,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
994,I013,I013-Polambakkam Govt School,Polambakkam Govt Higher Secondary School.,Steel Cupboard 6 1/2',NaN,NaN,NaN,Institutions,Article,Furnitures,...,Steel Cupboard,1,12390.0,1,0,120,A_Institutions,0,0,0
995,I002,"I002-Athivakkam,Panchayat School","Athivakkam,Panchayat Union Primary School.",Steel cupboard Glass Book Shelf,NaN,NaN,NaN,Institutions,Article,Furnitures,...,Glass Book Shelf,1,14750.0,1,0,121,A_Institutions,0,0,0
996,I012,I012-Avanippur Government School,Avanippur Government Higher Secondary School,Ceiling Fan,NaN,NaN,NaN,Institutions,Article,Electricals,...,Ceiling Fan,10,15000.0,10,0,122,A_Institutions,0,0,0
997,I013,I013-Polambakkam Govt School,Polambakkam Govt Higher Secondary School.,Ceiling Fan,NaN,NaN,NaN,Institutions,Article,Electricals,...,Ceiling Fan,5,7500.0,5,0,122,A_Institutions,0,0,0


In [14]:
# data = data[data["Token Quantity"]!=0].reset_index(drop=True)
# data["Start Token No."] = int(0)  # create new column called Start Token No.
# data["End Token No."] = int(0)   # create new column called End Token No.
# data.at[0,"Start Token No."] = 1  # initialise first token as 1
# data.at[0,"End Token No."] = data.at[0,"Token Quantity"] * data.at[0,"Start Token No."] # this first token for first line
# for i in range(1,len(data)):
#     data.at[i,"Start Token No."] = data.at[i-1,"End Token No."] + 1
#     data.at[i,"End Token No."] = data.at[i,"Token Quantity"] + data.at[i,"Start Token No."] -1
# data

##### We create a new column called `Token Print for ARTL` which will have the value 1 or 0. 1 being its needed for label, 0 being label not needed. this helps us avoid unnecessary printing of Article labels, where we dont print all of them (Eg. Projects, Aid dont have articles to stick labels, as those are banners or cheques).

##### Here in this code, we set 1 for all Articles  and 0 for Project and Aid.

In [15]:
data["Token Print for ARTL"] = int(0)
data.loc[data["Item Type"] == 'Article', 'Token Print for ARTL']  = 1
data.loc[data["Requested Item"] == 'Plant Sapling', 'Token Print for ARTL']  = 0
data.loc[data["Requested Item"] == 'Provision items', 'Token Print for ARTL']  = 0
data.loc[data["Requested Item"] == 'Aluminium holed rice strainer with Handle', 'Token Print for ARTL']  = 0
data.loc[data["Requested Item"] == 'Goat(1 Pair)', 'Token Print for ARTL']  = 0
data.loc[data["Requested Item"] == 'Ortho Caliper', 'Token Print for ARTL']  = 0
data.loc[data["Requested Item"] == 'Fishing Net', 'Token Print for ARTL']  = 0
data.loc[data["Requested Item"] == 'Grocery Items', 'Token Print for ARTL']  = 0
data.loc[data["Requested Item"] == 'Artificial leg', 'Token Print for ARTL']  = 0
data.loc[data["Requested Item"] == 'Cow & Calf', 'Token Print for ARTL']  = 0

data

,Application Number,Names,Beneficiary Name,Requested Item,Handicapped Status,Gender,Gender Category,Beneficiary Type,Item Type,Article Category,...,Requested Item Tk,Quantity,Total Value,Waiting Hall Quantity,Token Quantity,Sequence No,R_Names,Start Token No.,End Token No.,Token Print for ARTL
0,P435,P435 - B.Senthil Kumar,B.Senthil Kumar,Cow & Calf,No,Male,NaN,Public,Article,Livestock,...,Cattle,1,45000.0,0,1,1,AA_Public,1,1,0
1,P049,P049 - Valli.M,Valli.M,Goat(1 Pair),No,Female,Married,Public,Article,Livestock,...,Goat,1,10000.0,0,1,2,AA_Public,2,2,0
2,P123,P123 - Viruthambal.M,Viruthambal.M,Goat(1 Pair),Yes,Female,Married,Public,Article,Livestock,...,Goat,1,10000.0,0,1,2,AA_Public,3,3,0
3,P124,P124 - Madurai.D,Madurai.D,Goat(1 Pair),Yes,Male,NaN,Public,Article,Livestock,...,Goat,1,10000.0,0,1,2,AA_Public,4,4,0
4,P125,P125 - Mariyammal.M,Mariyammal.M,Goat(1 Pair),No,Female,Widow,Public,Article,Livestock,...,Goat,1,10000.0,0,1,2,AA_Public,5,5,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
994,I013,I013-Polambakkam Govt School,Polambakkam Govt Higher Secondary School.,Steel Cupboard 6 1/2',NaN,NaN,NaN,Institutions,Article,Furnitures,...,Steel Cupboard,1,12390.0,1,0,120,A_Institutions,0,0,1
995,I002,"I002-Athivakkam,Panchayat School","Athivakkam,Panchayat Union Primary School.",Steel cupboard Glass Book Shelf,NaN,NaN,NaN,Institutions,Article,Furnitures,...,Glass Book Shelf,1,14750.0,1,0,121,A_Institutions,0,0,1
996,I012,I012-Avanippur Government School,Avanippur Government Higher Secondary School,Ceiling Fan,NaN,NaN,NaN,Institutions,Article,Electricals,...,Ceiling Fan,10,15000.0,10,0,122,A_Institutions,0,0,1
997,I013,I013-Polambakkam Govt School,Polambakkam Govt Higher Secondary School.,Ceiling Fan,NaN,NaN,NaN,Institutions,Article,Electricals,...,Ceiling Fan,5,7500.0,5,0,122,A_Institutions,0,0,1


In [16]:
data[data["Token Print for ARTL"]==1]["Requested Item"].unique()

array(['Kirolskar Min T8 Dlx Power Weeder',
       'Handicapped Two Wheeler-100%', 'Handicapped Two Wheeler-25%',
       'Two wheeler_LS', 'Auto (3 wheeler) - Aid',
       'E Vehicle Load Carrier (Mahendra)', 'Back Load tricycle',
       'Front Load tricycle', 'Handmover Tricycle Delux',
       'Wheel Chair Folding', 'Wheel Chair Fixed ',
       'Battery Powered Wheel Chair NEOFLY', 'Hearing Aid ', 'Laptop',
       'Laptop HP Victus', 'Desktop i5', 'Desktop Computer',
       'HP Printer 126NW (Heavy, All in 1)', 'Epson L6370 Color Printer',
       'Epson Printer L3250 (Lite)', 'Canon 6030b',
       'Tab (TB311XU) 4G calling & WiFi  ', 'Agri Battery Sprayer',
       'Agri Power Sprayer 2S', 'Agri Power Sprayer 4S',
       'Aspee BOLO power sprayer', 'Agri Manual Sprayer',
       'Agri Manual Sprayer 13L', 'Bosch Electrician Kit 10 Re',
       'Bosch Electrician Kit 13 Re', 'Bosch Rotary Hammer GBH 220',
       'Bosch Demolition hammer DCA 11E', 'Gp Welding Machine Arc 200',
       'Elec

In [17]:
data[data["Token Print for ARTL"]==0]["Requested Item"].unique()

array(['Cow & Calf', 'Goat(1 Pair)', 'Plant Sapling', 'Free Marriage',
       'Artificial leg', 'Ortho Caliper', 'Grocery Items',
       'Ex Gratia Aid', 'Cash Award for Meritorious Students',
       'Education Aid', 'Cash Aid for Handicapped', 'Medical Aid',
       'Business Aid', 'Livelihood Aid', 'Construction Aid',
       'Renovation Aid', 'Financial Aid', 'Environment Aid',
       'Provision items', 'Fishing Net',
       'Aluminium holed rice strainer with Handle'], dtype=object)

In [18]:
data["Requested Item Tk"] = data["Requested Item Tk"].replace({"Wet Grinder Floor 2L": "Wet Grinder FLR 2L"})

##### Now, we add all the data together (Data with 0 and the Previous file with generated tokens) as single file and we save them as `Generated_tokens.xlsx` in a dedicated folder. `save_path` is the location where we save the file. `file_name` is the name of the file. Please change the path accordingly to your system within double quotes,excluding file name.

In [19]:
save_path = "/Users/aswathshakthi/PycharmProjects/MLMR/MNP26/Result/"
file_name = "2_Master_Records_Token.xlsx"
#
# combined = pd.concat([data, Zero_data], ignore_index=True)
#
# # keep custom Beneficiary order
# benef_order = ["Institutions", "Public", "District"]
# combined["Beneficiary Type"] = pd.Categorical(
#     combined["Beneficiary Type"], categories=benef_order, ordered=True
# )
#
# # same custom keys as before
# combined["_handicap_first"] = (
#     combined["Handicapped Status"].astype(str).str.strip().str.lower().eq("yes")
#     .map({True: 0, False: 1})
# )
#
# is_public_laptop = (
#     combined["Beneficiary Type"].astype(str).eq("Public")
#     & combined["Requested Item"].astype(str).str.contains("Laptop", case=False, na=False)
# )
# combined["_p116_top"] = 1
# combined.loc[
#     is_public_laptop & combined["Application Number"].astype(str).eq("P116"),
#     "_p116_top"
# ] = 0
#
# # final custom sort (add Start Token as tie-breaker)
# combined = combined.sort_values(
#     by=[
#         "Sequence No",
#         "Beneficiary Type",
#         "_p116_top",
#         "_handicap_first",
#         "Names",
#         "Start Token No."
#     ],
#     ascending=[True, True, True, True, True, True],
#     ignore_index=True
# ).drop(columns=["_p116_top", "_handicap_first"])
#
# combined.to_excel(save_path + file_name, index=False)
# combined
# # data = pd.concat([data,Zero_data]).sort_values(by=["Sequence No","Beneficiary Type","Names","Start Token No."]).reset_index(drop=True)
data.to_excel(save_path+file_name,index=False)
data

,Application Number,Names,Beneficiary Name,Requested Item,Handicapped Status,Gender,Gender Category,Beneficiary Type,Item Type,Article Category,...,Requested Item Tk,Quantity,Total Value,Waiting Hall Quantity,Token Quantity,Sequence No,R_Names,Start Token No.,End Token No.,Token Print for ARTL
0,P435,P435 - B.Senthil Kumar,B.Senthil Kumar,Cow & Calf,No,Male,NaN,Public,Article,Livestock,...,Cattle,1,45000.0,0,1,1,AA_Public,1,1,0
1,P049,P049 - Valli.M,Valli.M,Goat(1 Pair),No,Female,Married,Public,Article,Livestock,...,Goat,1,10000.0,0,1,2,AA_Public,2,2,0
2,P123,P123 - Viruthambal.M,Viruthambal.M,Goat(1 Pair),Yes,Female,Married,Public,Article,Livestock,...,Goat,1,10000.0,0,1,2,AA_Public,3,3,0
3,P124,P124 - Madurai.D,Madurai.D,Goat(1 Pair),Yes,Male,NaN,Public,Article,Livestock,...,Goat,1,10000.0,0,1,2,AA_Public,4,4,0
4,P125,P125 - Mariyammal.M,Mariyammal.M,Goat(1 Pair),No,Female,Widow,Public,Article,Livestock,...,Goat,1,10000.0,0,1,2,AA_Public,5,5,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
994,I013,I013-Polambakkam Govt School,Polambakkam Govt Higher Secondary School.,Steel Cupboard 6 1/2',NaN,NaN,NaN,Institutions,Article,Furnitures,...,Steel Cupboard,1,12390.0,1,0,120,A_Institutions,0,0,1
995,I002,"I002-Athivakkam,Panchayat School","Athivakkam,Panchayat Union Primary School.",Steel cupboard Glass Book Shelf,NaN,NaN,NaN,Institutions,Article,Furnitures,...,Glass Book Shelf,1,14750.0,1,0,121,A_Institutions,0,0,1
996,I012,I012-Avanippur Government School,Avanippur Government Higher Secondary School,Ceiling Fan,NaN,NaN,NaN,Institutions,Article,Electricals,...,Ceiling Fan,10,15000.0,10,0,122,A_Institutions,0,0,1
997,I013,I013-Polambakkam Govt School,Polambakkam Govt Higher Secondary School.,Ceiling Fan,NaN,NaN,NaN,Institutions,Article,Electricals,...,Ceiling Fan,5,7500.0,5,0,122,A_Institutions,0,0,1


### Label

From here we see how to Print the labels. First, we set the path of our file to use `result_path`, which has the path of `Generated_token.xlsx`. `label_path` is the path where we store the generated labels. the path can be change according to your system.

In [20]:
result_path = "/Users/aswathshakthi/PycharmProjects/MLMR/MNP26/Result/2_Master_Records_Token.xlsx"
label_path = "/Users/aswathshakthi/PycharmProjects/MLMR/MNP26/Token/Label/"

#### Now once the token generataed file is ready, we move to Label creation process. We might need Labels for following purposes
#### 1. Labels for posting in Articles, that are set to distribute in the MNP event Hall premise.

The labels come in two sizes: 2L and 12L.

For the 12L size:
- A4 sheet dimensions: 210mm x 297mm
- Top margin: 9mm
- Side margin: 4mm
- Vertical pitch: 47mm
- Horizontal pitch: 102mm
- Label height: 44mm
- Label width: 100mm
- Number across: 2
- Number down: 6
- Corner radius: 2mm

For the 2L size:
- A4 sheet dimensions: 210mm x 297mm
- Top margin: 2.5mm
- Side margin: 5mm
- Vertical pitch: 146mm
- Horizontal pitch: 200mm
- Label height: 146mm
- Label width: 200mm
- Number across: 1
- Number down: 2
- Corner radius: 2mm

The 12L labels are used for all articles except for larger articles such as tables, steel cupboards, etc., for which we use the 2L size. The labels contain the District/Public/Institution Names, Names of the Requested Item, and Token Number in a defined format.

1. **Reading and Filtering Data**:
   - The code begins by importing necessary libraries, such as `pandas` for data manipulation and `canvas` from `reportlab` for PDF generation.
   - It reads data from an Excel file (`result_path`) into a Pandas DataFrame named `Article_data`.
   - Filters are applied to this DataFrame based on specific conditions:
     - Articles with a non-zero "Change QTY" and requiring label printing (`Token Print for ARTL != 0`) are selected.
     - Certain articles (e.g., "Steel Cupboard", "Office Table 4x2", etc.) are excluded from the selection, has it requires 2L labels

2. **Data Formatting**:
   - The code modifies the case of values in the "Names" and "Requested Item" columns to title case.
   - It replaces specific article names to ensure consistent formatting.

3. **PDF Label Generation**:
   - A function named `generate_pdf(labels)` is defined to create labels in PDFs.
   - The function initializes parameters such as page size, margins, pitch, label dimensions, etc.
   - It iterates over each label in the provided data and determines its position on the PDF page based on the specified layout.
   - For each label, it formats and aligns the token number, district name, and Requested Item name appropriately.
   - The function handles pagination for printing, ensuring that labels for each article are printed contiguously with appropriate breaks( each article in new page).
   - The generated PDF file is saved with the specified name (`1.Article_Labels.pdf`) in the provided directory (`label_path`).


**Note**: This code prints label for all articles except bigger ones like "Steel Cupboard", "Office Table 4x2", "S Type Chair", "Reception 4 seater". in separate pages. (i.e. each new articles with take a separate pages continuously). Printing separately helps us post them on article without any confusion or missing. its also easy to distribute among peers for posting, saving time. But has the cons of wasting pages.
the `KFC` and `KFS` refers to `Keep for Continuous` and `Keep for Separate`. If we want to print them continuously without wasting pages, lets remove the #(also called uncommenting) in the front of the line where `KFC` mentioned. if separate printing is required, uncomment the line where `KFS` is mentioned but comment the `KFC` lines.

In [21]:
## Labels for posting in Articles - Seperate
## Read the data
Article_data = pd.read_excel(result_path)
Article_data = Article_data[
    (Article_data["Token Quantity"] != 0) &
    (Article_data["Token Print for ARTL"] != 0) &
    ~(Article_data["Requested Item"].isin([
        "Tiffen Set",
        "Tiffen Set + Alu Idli Box + MS Stove 2 Burner",
        "Tiffen Set + MS Stove 2 Burner",
        "Tiffen Set + Tea Can 10 Ltrs SS",
        "Push Cart Without Top",
        "Push Cart With Top",
        "Push Cart With Top + Alu Idli Box + MS Stove 2 Burner",
        "Office Table 4 X 2",
        "S Type Chair",
        "Steel Cupboard 6 1/2'",
    ]))
].reset_index(drop=True)


Article_data["Names"] = Article_data["Names"].str.title()
# Article_data["Requested Item"] = Article_data["Requested Item"].str.title()
# Article_data['Requested Item'] = Article_data['Requested Item'].str.replace('Sewing Machine Ord','Sewing Machine ORD')


# NOTE:  KFS - Keep for Separate , KFC - Keep for continuous
def generate_pdf(labels):
    c = canvas.Canvas(label_path+"1.Article_Labels_S.pdf", pagesize=portrait(A4))
    width, height = portrait(A4)
    top_margin = 9 * mm
    side_margin = 4 * mm
    vertical_pitch = 47 * mm
    horizontal_pitch = 102 * mm
    label_height = 44 * mm
    label_width = 100 * mm
    number_across = 2
    number_down = 6
    corner_radius = 2 * mm

    current_article = None  # Track the current article being printed. #KFS
    article_pages = 0  # Number of pages printed for the current article. #KFS

    for i, label in enumerate(labels["data"]):

        if label[2] != current_article: #KFS
            current_article = label[2] #KFS
            article_pages = 0  #KFS
            if i > 0: #KFS
                c.showPage()  #KFS

        if article_pages >= 12: #KFS
            c.showPage() #KFS
            article_pages = 0 #KFS


        # if i % (number_across * number_down) == 0:  #KFC
            # c.showPage()  # Start a new page every 12 labels #KFC
            # x_offset = side_margin #KFC
            # y_offset = height - top_margin #KFC

        col = article_pages % number_across # KFS
        row = article_pages // number_across % number_down #KFS
        # col = i % number_across  #KFC
        # row = i // number_across % number_down #KFC
        x = side_margin + col * horizontal_pitch
        y = height - top_margin - row * vertical_pitch - label_height

        #c.roundRect(x, y, label_width, label_height, corner_radius)

        token, district, article = label

        # Determine font size based on the length of the token number.
        token_font_size = 60
        if len(token) == 3:
            token_font_size = 50
        elif len(token) >= 4:
            token_font_size = 47


        style_token = ParagraphStyle(name='Arial', fontSize=token_font_size  , alignment=0)
        style_article = ParagraphStyle(name='Arial', fontSize=15, leading = 16, alignment=1) #leading  gives space between lines,
        style_district = ParagraphStyle(name='Arial', fontSize=15, leading = 14,spaceAfter=20, alignment=1)

        token_text = Paragraph(f"<b>{token}</b>", style_token)
        article_text = Paragraph(f"<b>{article}</b>", style_article)
        district_text = Paragraph(f"<b>{district}</b>", style_district)

        # Calculate positions for left alignment
        token_text_width, token_text_height = token_text.wrap(label_width / 2, label_height)
        token_text.drawOn(c, x + 8 * mm, y + 0.66 * (label_height - token_text_height)) #8mm for horizontal , adjust 0.66 for vertical

        # Calculate positions for right alignment
        district_text_width, district_text_height = district_text.wrap(label_width / 2, label_height)
        district_text.drawOn(c, x + 0.8 * (label_width - district_text_width), y + 0.85 * (label_height - district_text_height))

        # Calculate positions for right alignment
        article_text_width, article_text_height = article_text.wrap(label_width / 2, label_height)
        article_text.drawOn(c, x + 0.8 * (label_width - article_text_width), y + 6 * mm)

        article_pages += 1 #KFS
    c.save()


# Generate labels

labels_data = []
for _, row in Article_data.iterrows():
    tokens = range(row['Start Token No.'], row['End Token No.'] + 1)
    for token in tokens:
        label_text = (str(token), row['Names'], row['Requested Item Tk'])
        labels_data.append(label_text)

# Generate PDF file with labels
labels = {"data": labels_data}
generate_pdf(labels)

In [22]:
## Labels for posting in Articles  - Continous
## Read the data
Article_data = pd.read_excel(result_path)
Article_data = Article_data[
    (Article_data["Token Quantity"] != 0) &
    (Article_data["Token Print for ARTL"] != 0) &
    ~(Article_data["Requested Item"].isin([
        "Tiffen Set",
        "Tiffen Set + Alu Idli Box + MS Stove 2 Burner",
        "Tiffen Set + MS Stove 2 Burner",
        "Tiffen Set + Tea Can 10 Ltrs SS",
        "Push Cart Without Top",
        "Push Cart With Top",
        "Push Cart With Top + Alu Idli Box + MS Stove 2 Burner",
        "Office Table 4 X 2",
        "S Type Chair",
        "Steel Cupboard 6 1/2'",
    ]))
].reset_index(drop=True)

Article_data["Names"] = Article_data["Names"].str.title()
# Article_data["Requested Item"] = Article_data["Requested Item"].str.title()
# Article_data['Requested Item'] = Article_data['Requested Item'].str.replace('Sewing Machine Ord','Sewing Machine ORD')


# NOTE:  KFS - Keep for Separate , KFC - Keep for continuous
def generate_pdf(labels):
    c = canvas.Canvas(label_path+"1.Article_Labels_C.pdf", pagesize=portrait(A4))
    width, height = portrait(A4)
    top_margin = 9 * mm
    side_margin = 4 * mm
    vertical_pitch = 47 * mm
    horizontal_pitch = 102 * mm
    label_height = 44 * mm
    label_width = 100 * mm
    number_across = 2
    number_down = 6
    corner_radius = 2 * mm

    current_article = None  # Track the current article being printed. #KFS
    article_pages = 0  # Number of pages printed for the current article. #KFS

    for i, label in enumerate(labels["data"]):

        # if label[2] != current_article: #KFS
        #     current_article = label[2] #KFS
        #     article_pages = 0  #KFS
        #     if i > 0: #KFS
        #         c.showPage()  #KFS
        #
        # if article_pages >= 12: #KFS
        #     c.showPage() #KFS
        #     article_pages = 0 #KFS


        if i % (number_across * number_down) == 0:  #KFC
            c.showPage()  # Start a new page every 12 labels #KFC
            x_offset = side_margin #KFC
            y_offset = height - top_margin #KFC

        # col = article_pages % number_across # KFS
        # row = article_pages // number_across % number_down #KFS
        col = i % number_across  #KFC
        row = i // number_across % number_down #KFC
        x = side_margin + col * horizontal_pitch
        y = height - top_margin - row * vertical_pitch - label_height

        #c.roundRect(x, y, label_width, label_height, corner_radius)

        token, district, article = label

        # Determine font size based on the length of the token number.
        token_font_size = 60
        if len(token) == 3:
            token_font_size = 50
        elif len(token) >= 4:
            token_font_size = 47


        style_token = ParagraphStyle(name='Arial', fontSize=token_font_size  , alignment=0)
        style_article = ParagraphStyle(name='Arial', fontSize=15, leading = 16, alignment=1) #leading  gives space between lines,
        style_district = ParagraphStyle(name='Arial', fontSize=15, leading = 14,spaceAfter=20, alignment=1)

        token_text = Paragraph(f"<b>{token}</b>", style_token)
        article_text = Paragraph(f"<b>{article}</b>", style_article)
        district_text = Paragraph(f"<b>{district}</b>", style_district)

        # Calculate positions for left alignment
        token_text_width, token_text_height = token_text.wrap(label_width / 2, label_height)
        token_text.drawOn(c, x + 8 * mm, y + 0.66 * (label_height - token_text_height)) #8mm for horizontal , adjust 0.66 for vertical

        # Calculate positions for right alignment
        district_text_width, district_text_height = district_text.wrap(label_width / 2, label_height)
        district_text.drawOn(c, x + 0.8 * (label_width - district_text_width), y + 0.85 * (label_height - district_text_height))

        # Calculate positions for right alignment
        article_text_width, article_text_height = article_text.wrap(label_width / 2, label_height)
        article_text.drawOn(c, x + 0.8 * (label_width - article_text_width), y + 6 * mm)

        # article_pages += 1 #KFS
    c.save()


# Generate labels

labels_data = []
for _, row in Article_data.iterrows():
    tokens = range(row['Start Token No.'], row['End Token No.'] + 1)
    for token in tokens:
        label_text = (str(token), row['Names'], row['Requested Item Tk'])
        labels_data.append(label_text)

# Generate PDF file with labels
labels = {"data": labels_data}
generate_pdf(labels)

### 1.1. Chair Labels

In [23]:
## Labels for posting on Chair - Seperate - got info about items to print(ppl who are sitting in chair to receive articles) then in the generate token file modify the print (0/1) with new column
## Read the data
Article_data = pd.read_excel(result_path)
Article_data = Article_data[Article_data["Token Print for ARTL"] != 0].reset_index(drop=True)

# Article_data["Names"] = Article_data["Names"].str.title()
# Article_data["Requested Item"] = Article_data["Requested Item"].str.title()



# NOTE:  KFS - Keep for Separate , KFC - Keep for continuous
def generate_pdf(labels):
    c = canvas.Canvas(label_path+"Chair_Labels_S.pdf", pagesize=portrait(A4))
    width, height = portrait(A4)
    top_margin = 9 * mm
    side_margin = 4 * mm
    vertical_pitch = 47 * mm
    horizontal_pitch = 102 * mm
    label_height = 44 * mm
    label_width = 100 * mm
    number_across = 2
    number_down = 6
    corner_radius = 2 * mm

    current_article = None  # Track the current article being printed. #KFS
    article_pages = 0  # Number of pages printed for the current article. #KFS

    for i, label in enumerate(labels["data"]):

        if label[2] != current_article: #KFS
            current_article = label[2] #KFS
            article_pages = 0  #KFS
            if i > 0: #KFS
                c.showPage()  #KFS

        if article_pages >= 12: #KFS
            c.showPage() #KFS
            article_pages = 0 #KFS


        # if i % (number_across * number_down) == 0:  #KFC
            # c.showPage()  # Start a new page every 12 labels #KFC
            # x_offset = side_margin #KFC
            # y_offset = height - top_margin #KFC

        col = article_pages % number_across # KFS
        row = article_pages // number_across % number_down #KFS
        # col = i % number_across  #KFC
        # row = i // number_across % number_down #KFC
        x = side_margin + col * horizontal_pitch
        y = height - top_margin - row * vertical_pitch - label_height

        #c.roundRect(x, y, label_width, label_height, corner_radius)

        token, district, article = label

        # Determine font size based on the length of the token number.
        token_font_size = 60
        if len(token) == 3:
            token_font_size = 50
        elif len(token) >= 4:
            token_font_size = 47


        style_token = ParagraphStyle(name='Arial', fontSize=token_font_size  , alignment=0)
        style_article = ParagraphStyle(name='Arial', fontSize=15, leading = 16, alignment=1) #leading  gives space between lines,
        style_district = ParagraphStyle(name='Arial', fontSize=15, leading = 14,spaceAfter=20, alignment=1)

        token_text = Paragraph(f"<b>{token}</b>", style_token)
        article_text = Paragraph(f"<b>{article}</b>", style_article)
        district_text = Paragraph(f"<b>{district}</b>", style_district)

        # Calculate positions for left alignment
        token_text_width, token_text_height = token_text.wrap(label_width / 2, label_height)
        token_text.drawOn(c, x + 8 * mm, y + 0.66 * (label_height - token_text_height)) #8mm for horizontal , adjust 0.66 for vertical

        # Calculate positions for right alignment
        district_text_width, district_text_height = district_text.wrap(label_width / 2, label_height)
        district_text.drawOn(c, x + 0.8 * (label_width - district_text_width), y + 0.85 * (label_height - district_text_height))

        # Calculate positions for right alignment
        article_text_width, article_text_height = article_text.wrap(label_width / 2, label_height)
        article_text.drawOn(c, x + 0.8 * (label_width - article_text_width), y + 6 * mm)

        article_pages += 1 #KFS
    c.save()


# Generate labels

labels_data = []
for _, row in Article_data.iterrows():
    tokens = range(row['Start Token No.'], row['End Token No.'] + 1)
    for token in tokens:
        label_text = (str(token), row['Names'], row['Requested Item Tk'])
        labels_data.append(label_text)

# Generate PDF file with labels
labels = {"data": labels_data}
generate_pdf(labels)

In [24]:
## Labels for posting on Chair  - Continous
## Read the data
Article_data = pd.read_excel(result_path)
Article_data = Article_data[
    (Article_data["Token Quantity"] > 0)
].reset_index(drop=True)

Article_data["Names"] = Article_data["Names"].str.title()
# Article_data["Requested Item"] = Article_data["Requested Item"].str.title()


# NOTE:  KFS - Keep for Separate , KFC - Keep for continuous
def generate_pdf(labels):
    c = canvas.Canvas(label_path+"Chair_Labels_C.pdf", pagesize=portrait(A4))
    width, height = portrait(A4)
    top_margin = 9 * mm
    side_margin = 4 * mm
    vertical_pitch = 47 * mm
    horizontal_pitch = 102 * mm
    label_height = 44 * mm
    label_width = 100 * mm
    number_across = 2
    number_down = 6
    corner_radius = 2 * mm

    current_article = None  # Track the current article being printed. #KFS
    article_pages = 0  # Number of pages printed for the current article. #KFS

    for i, label in enumerate(labels["data"]):

        # if label[2] != current_article: #KFS
        #     current_article = label[2] #KFS
        #     article_pages = 0  #KFS
        #     if i > 0: #KFS
        #         c.showPage()  #KFS
        #
        # if article_pages >= 12: #KFS
        #     c.showPage() #KFS
        #     article_pages = 0 #KFS


        if i % (number_across * number_down) == 0:  #KFC
            c.showPage()  # Start a new page every 12 labels #KFC
            x_offset = side_margin #KFC
            y_offset = height - top_margin #KFC

        # col = article_pages % number_across # KFS
        # row = article_pages // number_across % number_down #KFS
        col = i % number_across  #KFC
        row = i // number_across % number_down #KFC
        x = side_margin + col * horizontal_pitch
        y = height - top_margin - row * vertical_pitch - label_height

        #c.roundRect(x, y, label_width, label_height, corner_radius)

        token, district, article = label

        # Determine font size based on the length of the token number.
        token_font_size = 60
        if len(token) == 3:
            token_font_size = 50
        elif len(token) >= 4:
            token_font_size = 47


        style_token = ParagraphStyle(name='Arial', fontSize=token_font_size  , alignment=0)
        style_article = ParagraphStyle(name='Arial', fontSize=15, leading = 16, alignment=1) #leading  gives space between lines,
        style_district = ParagraphStyle(name='Arial', fontSize=15, leading = 14,spaceAfter=20, alignment=1)

        token_text = Paragraph(f"<b>{token}</b>", style_token)
        article_text = Paragraph(f"<b>{article}</b>", style_article)
        district_text = Paragraph(f"<b>{district}</b>", style_district)

        # Calculate positions for left alignment
        token_text_width, token_text_height = token_text.wrap(label_width / 2, label_height)
        token_text.drawOn(c, x + 8 * mm, y + 0.66 * (label_height - token_text_height)) #8mm for horizontal , adjust 0.66 for vertical

        # Calculate positions for right alignment
        district_text_width, district_text_height = district_text.wrap(label_width / 2, label_height)
        district_text.drawOn(c, x + 0.8 * (label_width - district_text_width), y + 0.85 * (label_height - district_text_height))

        # Calculate positions for right alignment
        article_text_width, article_text_height = article_text.wrap(label_width / 2, label_height)
        article_text.drawOn(c, x + 0.8 * (label_width - article_text_width), y + 6 * mm)

        # article_pages += 1 #KFS
    c.save()


# Generate labels

labels_data = []
for _, row in Article_data.iterrows():
    tokens = range(row['Start Token No.'], row['End Token No.'] + 1)
    for token in tokens:
        label_text = (str(token), row['Names'], row['Requested Item Tk'])
        labels_data.append(label_text)

# Generate PDF file with labels
labels = {"data": labels_data}
generate_pdf(labels)

#### 2. Labels for Institution Articles in 2L
#### This code only takes the bigger articles requiring 2L sheets from the data and prints them accordingly.
The code segment generates PDF labels for specific articles and saves them at a specified location. Here's a detailed breakdown:

1. **File Reading and Preprocessing**:
   - Reads data from an Excel file located at `result_path` into a Pandas DataFrame named `TL_data`.
   - Filters the DataFrame to select rows where "Change QTY" is not equal to zero.
   - Converts the "Names" and "Requested Item" columns to title case.
   - Replaces specific article names to ensure consistent formatting.

2. **Filtering Data for Specific Articles**:
   - Further filters the data to select rows where the "Requested Item" matches certain predefined article names like "Steel Cupboard", "Office Table 4X2","S Type Chair", "Reception 4 Seater", creating a new DataFrame named `TL_fil_data`.

3. **PDF Label Generation Function**:
   - Defines a function named `generate_pdf(labels)` to create PDF labels for the filtered articles.
   - Initializes parameters such as page size, margins, pitch, label dimensions, etc of the 2L sheet.
   - Iterates over each label in the provided data and determines its position on the PDF page based on the specified layout.
   - Formats and aligns the token number, district name, and Requested Item name on each label.
   - Draws the labels on the PDF page, ensuring proper alignment and spacing.
   - The PDF file is saved at a location specified in the code, with the filename `"2.Article_2L_Labels.pdf"` within the `label_path` directory.


In [53]:
TL_data = pd.read_excel(result_path)
TL_data = TL_data[TL_data["Token Quantity"]!=0].reset_index(drop=True)
TL_data["Names"] = TL_data["Names"].str.title()
# TL_data["Requested Item"] = TL_data["Requested Item"].str.title()
# TL_data['Requested Item'] = TL_data['Requested Item'].str.replace('Sewing Machine Ord','Sewing Machine ORD')


TL_fil_data = TL_data[TL_data["Requested Item"].isin(["Tiffen Set",
        "Tiffen Set + Alu Idli Box + MS Stove 2 Burner",
        "Tiffen Set + MS Stove 2 Burner",
        "Tiffen Set + Tea Can 10 Ltrs SS",
        "Push Cart Without Top",
        "Push Cart With Top",
        "Push Cart With Top + Alu Idli Box + MS Stove 2 Burner",
        "Office Table 4 X 2",
        "S Type Chair",
        "Steel Cupboard 6 1/2'"])].reset_index(drop=True)


#"Pvc Chairs"? need small or big?
def generate_pdf(labels):
    c = canvas.Canvas(label_path+"2.Article_2L_Labels.pdf", pagesize=portrait(A4))
    width, height = portrait(A4)
    top_margin = 2.5 * mm
    side_margin = 5 * mm
    vertical_pitch = 146 * mm
    horizontal_pitch = 200 * mm
    label_height = 146 * mm
    label_width = 200 * mm
    number_across = 1
    number_down = 2
    corner_radius = 2 * mm

    for i, label in enumerate(labels["data"]):
        if i % (number_across * number_down) == 0:
            c.showPage()  # Start a new page every 12 labels
            x_offset = side_margin
            y_offset = height - top_margin

        col = i % number_across
        row = i // number_across % number_down
        x = x_offset + col * horizontal_pitch
        y = y_offset - row * vertical_pitch - label_height
        # c.roundRect(x, y, label_width, label_height, corner_radius)

        token, district, article = label

        # Determine font size based on the length of the token number
        token_font_size = 80
        if len(token) == 3:
            token_font_size = 80
        elif len(token) >= 4:
            token_font_size = 70


        style_token = ParagraphStyle(name='Arial', fontSize=token_font_size  , alignment=0)
        style_article = ParagraphStyle(name='Arial', fontSize=35, leading = 35, alignment=1) #leading  gives space between lines,
        style_district = ParagraphStyle(name='Arial', fontSize=35, leading = 35, alignment=1)

        token_text = Paragraph(f"<b>{token}</b>", style_token)
        article_text = Paragraph(f"<b>{article}</b>", style_article)
        district_text = Paragraph(f"<b>{district}</b>", style_district)

        # Calculate positions for left alignment
        token_text_width, token_text_height = token_text.wrap(label_width / 2, label_height)
        token_text.drawOn(c, x + 8 * mm, y + 0.66 * (label_height - token_text_height)) #8mm for horizontal , adjust 0.66 for vertical

        # Calculate positions for right alignment
        district_text_width, district_text_height = district_text.wrap(label_width / 2, label_height)
        district_text.drawOn(c, x + 0.8 * (label_width - district_text_width), y + 0.66 * (label_height - district_text_height)) # decrease y to move down

        # Calculate positions for right alignment
        article_text_width, article_text_height = article_text.wrap(label_width / 2, label_height)
        article_text.drawOn(c, x + 0.8 * (label_width - article_text_width), y + 35 * mm)#increase y to move up


    c.save()

# Generate labels

labels_data = []
for _, row in TL_fil_data.iterrows():
    tokens = range(row['Start Token No.'], row['End Token No.'] + 1)
    for token in tokens:
        label_text = (str(token), row['Names'], row['Requested Item Tk'])

        labels_data.append(label_text)

# Generate PDF file with labels
labels = {"data": labels_data}
generate_pdf(labels)

#### 3. Labels for posting in Beneficiary badges from District, who collect articles at event premise
For District, we print labels in the same format as articles and post them on an MNP Event badge. This way, beneficiaries can present the badge and receive the articles. Some beneficiaries prefer to receive them later from the storeroom. For those, we won't print labels. Those data are marked as `Token Quantity= 0`.We filter the data where `Token Quantitynot = 0` and records belong to `District` and print for them.


1. **Data Preparation:**
   - Reads data from an Excel file specified by `result_path`.
   - Filters the data to include only rows where the "Change QTY" column is not zero and has records of District.
   - Converts the "Names" and "Requested Item" columns to title case and replaces specific article names for consistent formatting.
   - Sorts the data by "Names" and "Start Token No." and stores it in `d1_data`.
   - Extracts district-specific data from `d1_data` and stores it in `district_data`.

2. **PDF Generation and Saving:**
   - Initializes parameters such as page size, margins, pitch, label dimensions, etc.
   - Iterates over each label in the provided data.
   - Tracks the current article being printed and the number of pages printed for the current article.
   - Checks if a new page should be started based on the number of labels printed for the current article.
   - Calculates the position for each label on the page based on the specified layout.
   - Draws the token number, district name, and article name on each label, adjusting font sizes and alignment as needed.
   - Generates labels for each district-related article by iterating over rows in `district_data`.
   - For each token within the specified range (`Start Token No.` to `End Token No.`), creates a label containing the token number, district name, and Requested Item name.
   - Calls the `generate_pdf()` function with the list of label data.
   - This triggers the PDF generation process, where labels for the district-related articles are drawn on each page according to the specified layout and content.
   - The resulting PDF file contains labels for the selected district-related articles.
   - The PDF file is saved in the directory specified by `label_path` with the filename `"3.District_Labels.pdf"`.

**Note**:
the `KFC` and `KFS` refers to `Keep for Continuous` and `Keep for Separate`. If we want to print them continuously without wasting pages, lets remove the #(also called uncommenting) in the front of the line where `KFC` mentioned. if separate printing is required, uncomment the line where `KFS` is mentioned but comment the `KFC` lines.

In [54]:
# Read the data
d_data = pd.read_excel(result_path)
d_data = d_data[d_data["Token Quantity"]!=0].reset_index(drop=True)
d_data["Names"] = d_data["Names"].str.title()
# d_data["Requested Item"] = d_data["Requested Item"].str.title()
# d_data['Requested Item'] = d_data['Requested Item'].str.replace('Sewing Machine Ord','Sewing Machine ORD')
d1_data = d_data.sort_values(by=["Names","Start Token No."]).reset_index(drop=True)
district_data = d1_data[d1_data["Beneficiary Type"]=="District"]


# NOTE:  KFS - Keep for Separate , KFC - Keep for continuous
def generate_pdf(labels):
    c = canvas.Canvas(label_path+"3.District_Labels.pdf", pagesize=portrait(A4))
    width, height = portrait(A4)
    top_margin = 9 * mm
    side_margin = 4 * mm
    vertical_pitch = 47 * mm
    horizontal_pitch = 102 * mm
    label_height = 44 * mm
    label_width = 100 * mm
    number_across = 2
    number_down = 6
    corner_radius = 2 * mm

    current_article = None  # Track the current article being printed. #KFS
    article_pages = 0  # Number of pages printed for the current article. #KFS

    for i, label in enumerate(labels["data"]):

        if label[1] != current_article:# #KFS
            current_article = label[1]#KFS
            article_pages = 0  #KFS
            if i > 0:  #KFS
                c.showPage()  #KFS

        if article_pages >= 12:  #KFS
            c.showPage()  #KFS
            article_pages = 0  #KFS


        # if i % (number_across * number_down) == 0:  #KFC
        #     c.showPage()  # Start a new page every 12 labels #KFC
        #     x_offset = side_margin #KFC
        #     y_offset = height - top_margin #KFC

        col = article_pages % number_across #KFS
        row = article_pages // number_across % number_down #KFS
        # col = i % number_across #KFC
        # row = i // number_across % number_down #KFC
        x = side_margin + col * horizontal_pitch
        y = height - top_margin - row * vertical_pitch - label_height

        #c.roundRect(x, y, label_width, label_height, corner_radius)

        token, district, article = label

        # Determine font size based on the length of the token number
        token_font_size = 60
        if len(token) == 3:
            token_font_size = 50
        elif len(token) >= 4:
            token_font_size = 47


        style_token = ParagraphStyle(name='Arial', fontSize=token_font_size  , alignment=0)
        style_article = ParagraphStyle(name='Arial', fontSize=15, leading = 16, alignment=1) #leading  gives space between lines,
        style_district = ParagraphStyle(name='Arial', fontSize=15, leading = 14,spaceAfter=20, alignment=1)

        token_text = Paragraph(f"<b>{token}</b>", style_token)
        article_text = Paragraph(f"<b>{article}</b>", style_article)
        district_text = Paragraph(f"<b>{district}</b>", style_district)

        # Calculate positions for left alignment
        token_text_width, token_text_height = token_text.wrap(label_width / 2, label_height)
        token_text.drawOn(c, x + 8 * mm, y + 0.66 * (label_height - token_text_height)) #8mm for horizontal , adjust 0.66 for vertical

        # Calculate positions for right alignment
        district_text_width, district_text_height = district_text.wrap(label_width / 2, label_height)
        district_text.drawOn(c, x + 0.8 * (label_width - district_text_width), y + 0.85 * (label_height - district_text_height))

        # Calculate positions for right alignment
        article_text_width, article_text_height = article_text.wrap(label_width / 2, label_height)
        article_text.drawOn(c, x + 0.8 * (label_width - article_text_width), y + 6 * mm)

        article_pages += 1 #KFS
    c.save()


# Generate labels
labels_data = []
for _, row in district_data.iterrows():
    tokens = range(row['Start Token No.'], row['End Token No.'] + 1)
    for token in tokens:
        label_text = (str(token), row['Names'], row['Requested Item Tk'])
        labels_data.append(label_text)

# Generate PDF file with labels
labels = {"data": labels_data}
generate_pdf(labels)


#### 3.1. Flood, Accident and Deceased

Label printing for districts with Flood Relief, Deceased and Accident

In [55]:
# # Read the data
# d_data = pd.read_excel(result_path)
# d_data = d_data[d_data["Token Quantity"]!=0].reset_index(drop=True)
# d_data["Names"] = d_data["Names"].str.title()
# d_data["Requested Item"] = d_data["Requested Item"].str.title()
# # d_data['Requested Item'] = d_data['Requested Item'].str.replace('Sewing Machine Ord','Sewing Machine ORD')
# d1_data = d_data.sort_values(by=["Names","Start Token No."]).reset_index(drop=True)
# district_data = d1_data[d1_data["Beneficiary Type"]=="Others"]
#
#
# # NOTE:  KFS - Keep for Separate , KFC - Keep for continuous
# def generate_pdf(labels):
#     c = canvas.Canvas(label_path+"3.1.Others_Labels.pdf", pagesize=portrait(A4))
#     width, height = portrait(A4)
#     top_margin = 9 * mm
#     side_margin = 4 * mm
#     vertical_pitch = 47 * mm
#     horizontal_pitch = 102 * mm
#     label_height = 44 * mm
#     label_width = 100 * mm
#     number_across = 2
#     number_down = 6
#     corner_radius = 2 * mm
#
#     # current_article = None  # Track the current article being printed. #KFS
#     # article_pages = 0  # Number of pages printed for the current article. #KFS
#
#     for i, label in enumerate(labels["data"]):
#
#         # if label[1] != current_article:# #KFS
#         #     current_article = label[1]#KFS
#         #     article_pages = 0  #KFS
#         #     if i > 0:  #KFS
#         #         c.showPage()  #KFS
#         #
#         # if article_pages >= 12:  #KFS
#         #     c.showPage()  #KFS
#         #     article_pages = 0  #KFS
#
#
#         if i % (number_across * number_down) == 0:  #KFC
#             c.showPage()  # Start a new page every 12 labels #KFC
#             x_offset = side_margin #KFC
#             y_offset = height - top_margin #KFC
#
#         # col = article_pages % number_across #KFS
#         # row = article_pages // number_across % number_down #KFS
#         col = i % number_across #KFC
#         row = i // number_across % number_down #KFC
#         x = side_margin + col * horizontal_pitch
#         y = height - top_margin - row * vertical_pitch - label_height
#
#         #c.roundRect(x, y, label_width, label_height, corner_radius)
#
#         token, district, article = label
#
#         # Determine font size based on the length of the token number
#         token_font_size = 60
#         if len(token) == 3:
#             token_font_size = 50
#         elif len(token) >= 4:
#             token_font_size = 47
#
#
#         style_token = ParagraphStyle(name='Arial', fontSize=token_font_size  , alignment=0)
#         style_article = ParagraphStyle(name='Arial', fontSize=15, leading = 16, alignment=1) #leading  gives space between lines,
#         style_district = ParagraphStyle(name='Arial', fontSize=15, leading = 14,spaceAfter=20, alignment=1)
#
#         token_text = Paragraph(f"<b>{token}</b>", style_token)
#         article_text = Paragraph(f"<b>{article}</b>", style_article)
#         district_text = Paragraph(f"<b>{district}</b>", style_district)
#
#         # Calculate positions for left alignment
#         token_text_width, token_text_height = token_text.wrap(label_width / 2, label_height)
#         token_text.drawOn(c, x + 8 * mm, y + 0.66 * (label_height - token_text_height)) #8mm for horizontal , adjust 0.66 for vertical
#
#         # Calculate positions for right alignment
#         district_text_width, district_text_height = district_text.wrap(label_width / 2, label_height)
#         district_text.drawOn(c, x + 0.8 * (label_width - district_text_width), y + 0.85 * (label_height - district_text_height))
#
#         # Calculate positions for right alignment
#         article_text_width, article_text_height = article_text.wrap(label_width / 2, label_height)
#         article_text.drawOn(c, x + 0.8 * (label_width - article_text_width), y + 6 * mm)
#
#         # article_pages += 1 #KFS
#     c.save()
#
#
# # Generate labels
# labels_data = []
# for _, row in district_data.iterrows():
#     tokens = range(row['Start Token No.'], row['End Token No.'] + 1)
#     for token in tokens:
#         label_text = (str(token), row['Names'], row['Requested Item Tk'])
#         labels_data.append(label_text)
#
# # Generate PDF file with labels
# labels = {"data": labels_data}
# generate_pdf(labels)
#


#### 4. Badges for Balance Articles to District
For Beneficiaries, especially District, who prefer to collect the articles at the Store room, we create a label which has District Names, Names of the `Requested Item` , `QTY`(instead of Token Number) in a defined format.

For this we use a New Excel sheet `District Article Balance Lists 2024.xlsx` which has the difference of quantity from real to change QTY. All other method follows the previous and the file is saved under the name `4.Bln_Article_to_Dist.pdf` in the `label_path` directory.

change the path in `b_path` where your file belongs and update the name of sheet in `Sht_name`

In [56]:
# b_path = "/Users/aswathshakthi/PycharmProjects/MNP24/Label/Final reports/"
# Sht_name = "Label balance article"
# bln_art_data = pd.read_excel(b_path+"District Article Balance Lists 2024.xlsx",sheet_name=Sht_name)
#
#
# def generate_pdf(labels):
#     c = canvas.Canvas(label_path+"4.Bln_Article_to_Dist.pdf", pagesize=portrait(A4))
#     width, height = portrait(A4)
#     top_margin = 9 * mm
#     side_margin = 4 * mm
#     vertical_pitch = 47 * mm
#     horizontal_pitch = 102 * mm
#     label_height = 44 * mm
#     label_width = 100 * mm
#     number_across = 2
#     number_down = 6
#     corner_radius = 2 * mm
#
#
#     for i, label in enumerate(labels["data"]):
#         if i % (number_across * number_down) == 0:
#             c.showPage()  # Start a new page every 12 labels
#             x_offset = side_margin
#             y_offset = height - top_margin
#
#         col = i % number_across
#         row = i // number_across % number_down
#         x = x_offset + col * horizontal_pitch
#         y = y_offset - row * vertical_pitch - label_height
#         #c.roundRect(x, y, label_width, label_height, corner_radius)
#
#         district, article, balance = label
#
#         # # Determine font size based on the length of the token number
#         # token_font_size = 60
#         # if len(token) == 3:
#         #     token_font_size = 50
#         # elif len(token) >= 4:
#         #     token_font_size = 47
#
#
#         style_token = ParagraphStyle(name='Arial', fontSize=18  , alignment=0)
#         style_article = ParagraphStyle(name='Arial', fontSize=15, leading = 16, alignment=0) #leading  gives space between lines,
#         style_district = ParagraphStyle(name='Arial', fontSize=40, leading = 16, alignment=1)
#
#         token_text = Paragraph(f"<b>{district}</b>", style_token)
#         article_text = Paragraph(f"<b>{article}</b>", style_article)
#         district_text = Paragraph(f"<b>{balance}</b>", style_district)
#
#         # Calculate positions for left alignment
#         token_text_width, token_text_height = token_text.wrap(label_width / 2, label_height)
#         token_text.drawOn(c, x + 6 * mm, y + 0.8 * (label_height - token_text_height)) #8mm for horizontal , adjust 0.66 for vertical
#
#         # Calculate positions for right alignment
#         district_text_width, district_text_height = district_text.wrap(label_width / 2, label_height)
#         district_text.drawOn(c, x + 0.8 * (label_width - district_text_width), y + 0.5 * (label_height - district_text_height))
#
#         # Calculate positions for right alignment
#         article_text_width, article_text_height = article_text.wrap(label_width / 2, label_height)
#         article_text.drawOn(c, x + 0.1 * (label_width - article_text_width), y + 6 * mm)
#
#
#     c.save()
#
# # Generate labels
#
# labels_data = []
# for i in range(0,len(bln_art_data)):
#
#     label_text = (bln_art_data["Names"][i], bln_art_data["Requested Item"][i], str(bln_art_data["Balance"][i]))
#
#     labels_data.append(label_text)
#
# # Generate PDF file with labels
# labels = {"data": labels_data}
# generate_pdf(labels)

#### 5. Labels for Public & institute Badges
For Public and Institutions, we print labels in the same format as District and post them on an MNP Event badge. This way, beneficiaries can present the badge and receive the articles.The labels are printed in continous order as it will be distributed individually unlike District.We filter the data where `Token Quantitynot = 0` and records belong to `Public` and `Insititution` and print  them.

the files are saved under the name `5.PI_labels.pdf` in the `label_path` directory.

In [57]:
pidata = pd.read_excel(result_path)
pidata = pidata[pidata["Token Quantity"]!=0].reset_index(drop=True)
pidata["Names"] = pidata["Names"].str.title()
# pidata["Requested Item"] = pidata["Requested Item"].str.title()
# pidata['Requested Item'] = pidata['Requested Item'].str.replace('Sewing Machine Ord','Sewing Machine ORD')
pidata1 = pidata.sort_values(by=["Requested Item","Start Token No."]).reset_index(drop=True)
pi_data = pidata1[(pidata1["Beneficiary Type"]=="Institutions")]
pi_data

,Application Number,Names,Beneficiary Name,Requested Item,Handicapped Status,Gender,Gender Category,Beneficiary Type,Item Type,Article Category,...,Requested Item Tk,Quantity,Total Value,Waiting Hall Quantity,Token Quantity,Sequence No,R_Names,Start Token No.,End Token No.,Token Print for ARTL
5,I014,I014 - Adhiparasakthi Annai Illam,Adhiparasakthi Annai Illam,Agri Power Sprayer 2S,NaN,NaN,NaN,Institutions,Article,Agriculture,...,Agri Power Sprayer 2S,1,11025.00,0,1,44,A_Institutions,564,564,1
13,I001,"I001-Govt Leprosy Centre,Cgl","Government Leprosy Centre,Chengalpattu.",Aluminium holed rice strainer with Handle,NaN,NaN,NaN,Institutions,Article,Kitchen Items,...,Holed Rice Strainer,1,3500.00,0,1,114,A_Institutions,1240,1240,0
16,I014,I014 - Adhiparasakthi Annai Illam,Adhiparasakthi Annai Illam,Auto (3 wheeler) - Aid,NaN,NaN,NaN,Institutions,Article,Automotive,...,Auto (3 wheeler),1,175000.00,0,1,9,A_Institutions,26,26,1
92,I018,I018 - Adhiparasakthi Trust Institutions,Adhiparasakthi Trust Institutions,Cash Award for Meritorious Students,NaN,NaN,NaN,Institutions,Aid,Aid,...,Meritorious Reward,87,2175000.00,0,87,32,A_Institutions,261,347,0
104,I001,"I001-Govt Leprosy Centre,Cgl","Government Leprosy Centre,Chengalpattu.",Cooker 20 Litres,NaN,NaN,NaN,Institutions,Article,Kitchen Items,...,Cooker 20 Litres,1,5200.00,0,1,116,A_Institutions,1241,1241,1
106,I014,I014 - Adhiparasakthi Annai Illam,Adhiparasakthi Annai Illam,Desktop Computer,NaN,NaN,NaN,Institutions,Article,Computers & Printers,...,Desktop Computer,2,127000.00,0,2,25,A_Institutions,175,176,1
107,I016,I016 - Tashildar Kalavi,Tashildar Kalavi,Desktop Computer,NaN,NaN,NaN,Institutions,Article,Computers & Printers,...,Desktop Computer,1,63500.00,0,1,25,A_Institutions,177,177,1
113,I020,I020 - Aanmiga Makkal Thondu Iyakkam,Aanmiga makkal Thondu Iyakkam,Desktop i5,NaN,NaN,NaN,Institutions,Article,Computers & Printers,...,Desktop i5,1,255400.00,0,1,24,A_Institutions,174,174,1
115,I015,I015 - Gb Public School,GB Public School,Education Aid,NaN,NaN,NaN,Institutions,Aid,Aid,...,Education Aid,4,158000.00,0,4,33,A_Institutions,348,351,0
194,I019,I019 - Gem Of India,GEM OF INDIA,Environment Aid,NaN,NaN,NaN,Institutions,Aid,Environment,...,Environment Aid,1,30000.00,0,1,41,A_Institutions,555,555,0


In [60]:
# Function to generate PDF file with labels
# Read the data
pidata = pd.read_excel(result_path)
pidata = pidata[pidata["Token Quantity"]!=0].reset_index(drop=True)
pidata["Names"] = pidata["Names"].str.title()
# pidata["Requested Item"] = pidata["Requested Item"].str.title()
# pidata['Requested Item'] = pidata['Requested Item'].str.replace('Sewing Machine Ord','Sewing Machine ORD')
pidata1 = pidata.sort_values(by=["Names","Start Token No."]).reset_index(drop=True)
pi_data = pidata1[(pidata1["Beneficiary Type"]=="Institutions")]

def generate_pdf(labels):
    c = canvas.Canvas(label_path+"5.I_labels.pdf", pagesize=portrait(A4))
    width, height = portrait(A4)
    top_margin = 9 * mm
    side_margin = 4 * mm
    vertical_pitch = 47 * mm
    horizontal_pitch = 102 * mm
    label_height = 44 * mm
    label_width = 100 * mm
    number_across = 2
    number_down = 6
    corner_radius = 2 * mm


    for i, label in enumerate(labels["data"]):
        if i % (number_across * number_down) == 0:
            c.showPage()  # Start a new page every 12 labels
            x_offset = side_margin
            y_offset = height - top_margin

        col = i % number_across
        row = i // number_across % number_down
        x = x_offset + col * horizontal_pitch
        y = y_offset - row * vertical_pitch - label_height
        #c.roundRect(x, y, label_width, label_height, corner_radius)

        token, district, article = label

        # Determine font size based on the length of the token number
        token_font_size = 60
        if len(token) == 3:
            token_font_size = 50
        elif len(token) >= 4:
            token_font_size = 47


        style_token = ParagraphStyle(name='Arial', fontSize=token_font_size  , alignment=0)
        style_article = ParagraphStyle(name='Arial', fontSize=15, leading = 16, alignment=1) #leading  gives space between lines,
        style_district = ParagraphStyle(name='Arial', fontSize=15, leading = 14,spaceAfter=20, alignment=1)

        token_text = Paragraph(f"<b>{token}</b>", style_token)
        article_text = Paragraph(f"<b>{article}</b>", style_article)
        district_text = Paragraph(f"<b>{district}</b>", style_district)

        # Calculate positions for left alignment
        token_text_width, token_text_height = token_text.wrap(label_width / 2, label_height)
        token_text.drawOn(c, x + 8 * mm, y + 0.66 * (label_height - token_text_height)) #8mm for horizontal , adjust 0.66 for vertical

        # Calculate positions for right alignment
        district_text_width, district_text_height = district_text.wrap(label_width / 2, label_height)
        district_text.drawOn(c, x + 0.8 * (label_width - district_text_width), y + 0.85 * (label_height - district_text_height))

        # Calculate positions for right alignment
        article_text_width, article_text_height = article_text.wrap(label_width / 2, label_height)
        article_text.drawOn(c, x + 0.8 * (label_width - article_text_width), y + 6 * mm)


    c.save()

# Generate labels
labels_data = []
for _, row in pi_data.iterrows():
    tokens = range(row['Start Token No.'], row['End Token No.'] + 1)
    for token in tokens:
        label_text = (str(token), row['Names'], row['Requested Item Tk'])
        labels_data.append(label_text)

# Generate PDF file with labels
labels = {"data": labels_data}
generate_pdf(labels)

#### Check if tokens are printed correctly for both articles and badges
here we simple do a cross check to verify if we have printed all the tokens for articles, district, public, institutions.

In [496]:
print("Sum of Tokens Printed for Article : ",Article_data["Token Quantity"].sum() + TL_fil_data["Token Quantity"].sum())

print("Sum of Tokens Printed for District, Public, Institute (including Projects, aid): ",district_data["Token Quantity"].sum() + pi_data["Token Quantity"].sum() )
print("Sum of Tokens Printed for District, Public, Institute (only Projects, aid): ",district_data[district_data["Item Type"]!="Article"]["Token Quantity"].sum() + pi_data[pi_data["Item Type"]!="Article"]["Token Quantity"].sum() )
print("Difference After: ",(district_data["Token Quantity"].sum() + pi_data["Token Quantity"].sum()) - (district_data[district_data["Item Type"]!="Article"]["Token Quantity"].sum() + pi_data[pi_data["Item Type"]!="Article"]["Token Quantity"].sum()))

Sum of Tokens Printed for Article :  982
Sum of Tokens Printed for District, Public, Institute (including Projects, aid):  1259
Sum of Tokens Printed for District, Public, Institute (only Projects, aid):  300
Difference After:  959


#### 6. Seating Label for Project
Other Labels such as for Projects , which has the Serial number and "Project" printed on it. its used for Beneficiaries who get welfare Projects. we can print them manually by creating a excel file and inputting the values, using the label tools in excel. Alternatively, we can also do here. from the `Generated_token.xlsx` file,
we get the Projects and their quantity and we print them in the range, finally saving at `6.Project_label.pdf` of the `label_path`

In [497]:
# # Function to generate PDF file with labels
#
# seating = pd.read_excel("/Users/aswathshakthi/PycharmProjects/MNP25/Token/public_aid.xlsx")
# # seating = seating[seating["ITEM TYPE"]=="Project"]
# seat_data = pd.DataFrame({'Serial Number': range(1, seating['Token Quantity'].sum() + 1),
#                          'Project': ['Project'] * seating['Token Quantity'].sum()})
#
#
#
# def generate_pdf(labels):
#     c = canvas.Canvas(label_path+"6.Project_label.pdf", pagesize=portrait(A4))
#     width, height = portrait(A4)
#     top_margin = 9 * mm
#     side_margin = 4 * mm
#     vertical_pitch = 47 * mm
#     horizontal_pitch = 102 * mm
#     label_height = 44 * mm
#     label_width = 100 * mm
#     number_across = 2
#     number_down = 6
#     corner_radius = 2 * mm
#
#
#     for i, label in enumerate(labels["data"]):
#         if i % (number_across * number_down) == 0:
#             c.showPage()  # Start a new page every 12 labels
#             x_offset = side_margin
#             y_offset = height - top_margin
#
#         col = i % number_across
#         row = i // number_across % number_down
#         x = x_offset + col * horizontal_pitch
#         y = y_offset - row * vertical_pitch - label_height
#         #c.roundRect(x, y, label_width, label_height, corner_radius)
#
#         district, balance = label
#
#         # # Determine font size based on the length of the token number
#         # token_font_size = 60
#         # if len(token) == 3:
#         #     token_font_size = 50
#         # elif len(token) >= 4:
#         #     token_font_size = 47
#
#
#         style_token = ParagraphStyle(name='Arial', fontSize=45  , alignment=0)
#         style_article = ParagraphStyle(name='Arial', fontSize=15, leading = 16, alignment=0) #leading  gives space between lines,
#         style_district = ParagraphStyle(name='Arial', fontSize=30, leading = 14, alignment=1)
#
#         token_text = Paragraph(f"<b>{district}</b>", style_token)
#         district_text = Paragraph(f"<b>{balance}</b>", style_district)
#
#         # Calculate positions for left alignment
#         token_text_width, token_text_height = token_text.wrap(label_width / 2, label_height)
#         token_text.drawOn(c, x + 6 * mm, y + 0.8 * (label_height - token_text_height)) #8mm for horizontal , adjust 0.66 for vertical
#
#         # Calculate positions for right alignment
#         district_text_width, district_text_height = district_text.wrap(label_width / 2, label_height)
#         district_text.drawOn(c, x + 0.8 * (label_width - district_text_width), y + 0.69 * (label_height - district_text_height))
#
#
#
#     c.save()
#
# # Generate labels
#
# labels_data = []
# for i in range(0,len(seat_data)):
#
#     label_text = (seat_data["Serial Number"][i], seat_data["Project"][i])
#     labels_data.append(label_text)
#
# # Generate PDF file with labels
# labels = {"data": labels_data}
# generate_pdf(labels)

###### * These instructions are tailored for users with limited technical expertise. They adhere to the operating procedures followed in MNP 2024, and future changes may necessitate corresponding adjustments. Feel free to modify and adapt this notebook to suit your specific requirements and workflows. For any assistance or issues, please reach out to the notebook author.